In [ ]:
# Задача - обучить модель отслеживать точку взгляда пользователя на экран, использую данные камеры
# Результаты(2/5):

#	model							rmse_x		rmse_y
# 0	Poly2(Linear)					91.505097	63.785543
# 1	LightGBM						162.718544	58.884171
# 2	ElasticNet(alpha=0.1,l1=0.5)	124.723513	110.648618
# 3	Ridge(alpha=1.0)				124.555242	110.157764
# 4	LinearRegression				124.574788	110.156958
# 5	RandomForest(n=400)				216.464991	71.034076
# 6	Poly3(Linear)					250.121388	140.484191


In [17]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time
import os
import pandas as pd
from openpyxl import Workbook

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from lightgbm import LGBMRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# -----------------------------
# Параметры камеры и холста
# -----------------------------
CAMERA_WIDTH = 1920
CAMERA_HEIGHT = 1080
CAMERA_FPS = 30  # fps для захвата
CANVAS_SCALE = 1.0

# -----------------------------
# Настройка Mediapipe Face Mesh
# -----------------------------
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# -----------------------------
# Индексы ключевых точек
# -----------------------------
RIGHT_EYE = {'left': 33, 'right': 133, 'pupil': 468, 'iris':[468,469,470,471,472]}
LEFT_EYE  = {'left': 362, 'right': 263, 'pupil': 473, 'iris':[473,474,475,476,477]}
NOSE = {'top': 6, 'tip': 1}
MOUTH = {'left': 61, 'right': 291, 'upper': 13, 'lower': 14}

I0000 00:00:1768464996.239507       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [3]:
# -----------------------------
# Вспомогательные функции
# -----------------------------
def get_face_keypoints(face_landmarks, w, h):
    kp = {}
    kp['left_eye'] = {k:(int(face_landmarks.landmark[LEFT_EYE[k]].x * w),
                         int(face_landmarks.landmark[LEFT_EYE[k]].y * h)) for k in ['left','right','pupil']}
    kp['right_eye'] = {k:(int(face_landmarks.landmark[RIGHT_EYE[k]].x * w),
                          int(face_landmarks.landmark[RIGHT_EYE[k]].y * h)) for k in ['left','right','pupil']}
    kp['nose'] = [(int(face_landmarks.landmark[NOSE['top']].x * w),
                   int(face_landmarks.landmark[NOSE['top']].y * h)),
                  (int(face_landmarks.landmark[NOSE['tip']].x * w),
                   int(face_landmarks.landmark[NOSE['tip']].y * h))]
    kp['mouth'] = [(int(face_landmarks.landmark[MOUTH[k]].x * w),
                    int(face_landmarks.landmark[MOUTH[k]].y * h)) for k in ['left','right']]
    kp['upper_lip'] = (int(face_landmarks.landmark[MOUTH['upper']].x * w),
                       int(face_landmarks.landmark[MOUTH['upper']].y * h))
    kp['lower_lip'] = (int(face_landmarks.landmark[MOUTH['lower']].x * w),
                       int(face_landmarks.landmark[MOUTH['lower']].y * h))
    return kp


def get_pupil_center(face_landmarks, eye_dict, w, h):
    # Возвращает координаты z нормализованные по расстоянию между глазами
    xs = [face_landmarks.landmark[idx].x * w for idx in eye_dict['iris']]
    ys = [face_landmarks.landmark[idx].y * h for idx in eye_dict['iris']]
    zs = [face_landmarks.landmark[idx].z for idx in eye_dict['iris']]
    x_center = sum(xs)/len(xs)
    y_center = sum(ys)/len(ys)
    z_center = sum(zs)/len(zs)
    return x_center, y_center, z_center

def euler_from_rotation_matrix(rmat):
    sy = math.sqrt(rmat[0,0]**2 + rmat[1,0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = math.degrees(math.atan2(rmat[2,1], rmat[2,2]))
        yaw   = math.degrees(math.atan2(-rmat[2,0], sy))
        roll  = math.degrees(math.atan2(rmat[1,0], rmat[0,0]))
    else:
        pitch = math.degrees(math.atan2(-rmat[1,2], rmat[1,1]))
        yaw   = math.degrees(math.atan2(-rmat[2,0], sy))
        roll  = 0
    return pitch, yaw, roll

def get_image_points(face_landmarks, w, h):
    FACE_3D_IDX = {
        "nose_tip": 1,
        "chin": 152,
        "left_eye_corner": 263,
        "right_eye_corner": 33,
        "left_mouth_corner": 291,
        "right_mouth_corner": 61
    }
    points = []
    for idx in FACE_3D_IDX.values():
        lm = face_landmarks.landmark[idx]
        points.append((lm.x * w, lm.y * h))
    return np.array(points, dtype=np.float64)

def estimate_head_pose(face_landmarks, w, h):
    model_points = np.array([
        [0.0, 0.0, 0.0],          
        [0.0, -63.6, -12.5],      
        [-43.3, 32.7, -26.0],     
        [43.3, 32.7, -26.0],      
        [-28.9, -28.9, -24.1],    
        [28.9, -28.9, -24.1]      
    ], dtype=np.float64)
    image_points = get_image_points(face_landmarks, w, h)
    focal_length = w
    center = (w/2, h/2)
    camera_matrix = np.array([[focal_length, 0, center[0]],
                              [0, focal_length, center[1]],
                              [0, 0, 1]], dtype=np.float64)
    dist_coeffs = np.zeros((4,1))
    success, rotation_vector, _ = cv2.solvePnP(model_points, image_points, camera_matrix, dist_coeffs)
    rmat, _ = cv2.Rodrigues(rotation_vector)
    return euler_from_rotation_matrix(rmat)

def draw_face_keypoints(canvas, kp):
    for pt in [kp['left_eye']['left'], kp['left_eye']['right'], kp['left_eye']['pupil'],
               kp['right_eye']['left'], kp['right_eye']['right'], kp['right_eye']['pupil']]:
        cv2.circle(canvas, pt, 5, (255,0,0), -1)
    for pt in kp['nose']:
        cv2.circle(canvas, pt, 5, (255,0,0), -1)
    for pt in kp['mouth'] + [kp['upper_lip'], kp['lower_lip']]:
        cv2.circle(canvas, pt, 5, (255,0,0), -1)

def draw_eye_iris(canvas, pupil_center, eye_left, eye_right):
    eye_left = np.array(eye_left)
    eye_right = np.array(eye_right)
    radius = int(0.3 * np.linalg.norm(eye_right - eye_left))
    cv2.circle(canvas, (int(pupil_center[0]), int(pupil_center[1])), radius, (0,255,0), 2)

# -----------------------------
# Функция определения положения головы
# -----------------------------

def estimate_head_pose_3d_old(face_landmarks):
    """
    Оценивает pitch, yaw, roll головы, используя 3D landmarks Mediapipe.
    НЕ использует solvePnP.
    Устойчива к смещению головы относительно камеры.
    Возвращает углы в градусах.
    """

    def lm(idx):
        p = face_landmarks.landmark[idx]
        return np.array([p.x, p.y, p.z])

    # --- Опорные точки ---
    nose = lm(1)
    chin = lm(152)
    forehead = lm(10)
    left_cheek = lm(234)
    right_cheek = lm(454)
    left_eye = lm(33)
    right_eye = lm(263)

    # --- Оси головы ---
    # Вертикальная ось (вверх головы)
    vertical = forehead - chin
    vertical /= np.linalg.norm(vertical)

    # Горизонтальная ось (лево → право)
    horizontal = right_cheek - left_cheek
    horizontal /= np.linalg.norm(horizontal)

    # Ось "вперёд" (куда смотрит лицо)
    forward = np.cross(horizontal, vertical)
    forward /= np.linalg.norm(forward)

    # --- YAW (поворот влево/вправо) ---
    yaw = np.degrees(np.arctan2(forward[0], forward[2]))

    # --- PITCH (вверх/вниз) ---
    pitch = np.degrees(np.arctan2(forward[1], forward[2]))

    # --- ROLL (наклон головы) ---
    eye_line = right_eye - left_eye
    roll = np.degrees(np.arctan2(eye_line[1], eye_line[0]))

    return pitch, yaw, roll


def compute_eye_gaze_vector(pupil, eye_left, eye_right, strength=1.0):
    """
    pupil, eye_left, eye_right — np.array([x,y,z])
    возвращает 3D-вектор взгляда глаза
    """
    eye_center = (eye_left + eye_right) / 2
    eye_width = np.linalg.norm(eye_right - eye_left)

    if eye_width < 1e-6:
        return np.zeros(3)

    offset = (pupil - eye_center) / eye_width
    gaze = np.array([
        offset[0] * strength,   # left / right
        offset[1] * strength,   # up / down
        0.0
    ])
    return gaze
    
def draw_real_gaze(
    canvas,
    face_landmarks,
    rmat,
    avg_pupil,
    kp,
    scale=200,
    eye_strength=2.5
):
    h, w, _ = canvas.shape

    # --- центр головы (между глазами) ---
    origin = np.mean(
        [kp['left_eye']['pupil'], kp['right_eye']['pupil']],
        axis=0
    ).astype(int)

    # --- направление головы ---
    head_forward = rmat @ np.array([0, 0, 1])
    head_forward /= np.linalg.norm(head_forward)

    # --- 3D координаты глаз ---
    def lm3d(idx):
        lm = face_landmarks.landmark[idx]
        return np.array([lm.x * w, lm.y * h, lm.z])

    # левый глаз
    left_eye_gaze = compute_eye_gaze_vector(
        pupil=avg_pupil['left'],
        eye_left=lm3d(362),
        eye_right=lm3d(263),
        strength=eye_strength
    )

    # правый глаз
    right_eye_gaze = compute_eye_gaze_vector(
        pupil=avg_pupil['right'],
        eye_left=lm3d(33),
        eye_right=lm3d(133),
        strength=eye_strength
    )

    # --- комбинируем глаза ---
    eye_gaze = (left_eye_gaze + right_eye_gaze) / 2

    # --- итоговый вектор взгляда ---
    gaze_vector = head_forward + eye_gaze
    gaze_vector /= np.linalg.norm(gaze_vector)

    # --- проекция в 2D ---
    gaze_2d = gaze_vector[:2] * scale

    end = (origin + gaze_2d).astype(int)

    cv2.arrowedLine(
        canvas,
        tuple(origin),
        tuple(end),
        (0, 255, 0),
        3,
        tipLength=0.2
    )


def landmark_to_pixel(landmark, w, h):
    return int(landmark.x * w), int(landmark.y * h)


def iris_center(landmarks, iris_ids, w, h):
    xs, ys = [], []
    for idx in iris_ids:
        x, y = landmark_to_pixel(landmarks[idx], w, h)
        xs.append(x)
        ys.append(y)
    return int(np.mean(xs)), int(np.mean(ys))


def face_bbox(landmarks, ids, w, h):
    pts = [landmark_to_pixel(landmarks[i], w, h) for i in ids]
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return min(xs), min(ys), max(xs), max(ys)


# -----------------------------
# Фильтры Калмана
# -----------------------------

class Kalman3D:
    """
    3D Фильтр Калман (x, y, z).
    state = [x, y, z, x', y', z']  # позиция и скорость
    """
    def __init__(self, dt=1.0, process_var=1e-4, meas_var=1e-2):
        self.dt = dt
        # Матрица перехода состояния
        self.A = np.array([
            [1, 0, 0, dt, 0, 0],
            [0, 1, 0, 0, dt, 0],
            [0, 0, 1, 0, 0, dt],
            [0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 1]
        ], dtype=np.float64)
        # Матрица наблюдения
        self.H = np.array([
            [1, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0]
        ], dtype=np.float64)
        # Ковариации процесса и измерений
        self.Q = process_var * np.eye(6)
        self.R = meas_var * np.eye(3)
        self.P = np.eye(6)
        self.x = np.zeros((6, 1))  # начальное состояние

    def update(self, measurement):
        z = np.array(measurement).reshape((3,1))
        # --- Predict ---
        self.x = self.A @ self.x
        self.P = self.A @ self.P @ self.A.T + self.Q
        # --- Update ---
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(6) - K @ self.H) @ self.P
        return self.x[:3].flatten()  # возвращаем только позицию


class Kalman1D:
    """
    1D фильтр Калмана
    state = [k, k'] # позиция и скорость
    """
    def __init__(self, dt=1.0, process_var=1e-4, meas_var=1e-2):
        self.dt = dt
        # Матрица перехода состояния
        self.A = np.array([
            [1, dt],
            [0, 1]
        ], dtype=np.float64)
        # Матрица наблюдения
        self.H = np.array([[1, 0]], dtype=np.float64)
        # Ковариации процесса и измерений
        self.Q = process_var * np.eye(2)
        self.R = np.array([[meas_var]])
        self.P = np.eye(2)
        self.x = np.zeros((2,1))  # начальное состояние

    def update(self, measurement):
        z = np.array([[measurement]])
        # --- Predict ---
        self.x = self.A @ self.x
        self.P = self.A @ self.P @ self.A.T + self.Q
        # --- Update ---
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(2) - K @ self.H) @ self.P
        return self.x[0,0]  # возвращаем позицию


In [4]:
# -----------------------------
# Функция обработки изображения и определения координат ключевых точек и углов
# -----------------------------

def estimate_head_pose_3d(face_landmarks, prev_angles=None, alpha=0.3):
    """
    Оценка углов головы с учётом поворота и положения лица в кадре.
    Возвращает pitch, yaw, roll в градусах.
    """
    # 3D модель головы (мм)
    model_points = np.array([
        [0.0, 0.0, 0.0],        # нос
        [0.0, -63.6, -12.5],    # подбородок
        [-43.3, 32.7, -26.0],   # левый глаз
        [43.3, 32.7, -26.0],    # правый глаз
        [-28.9, -28.9, -24.1],  # левый рот
        [28.9, -28.9, -24.1]    # правый рот
    ], dtype=np.float64)

    # 2D точки
    image_points = np.array([
        [face_landmarks.landmark[1].x * CAMERA_WIDTH, face_landmarks.landmark[1].y * CAMERA_HEIGHT],
        [face_landmarks.landmark[152].x * CAMERA_WIDTH, face_landmarks.landmark[152].y * CAMERA_HEIGHT],
        [face_landmarks.landmark[263].x * CAMERA_WIDTH, face_landmarks.landmark[263].y * CAMERA_HEIGHT],
        [face_landmarks.landmark[33].x * CAMERA_WIDTH, face_landmarks.landmark[33].y * CAMERA_HEIGHT],
        [face_landmarks.landmark[291].x * CAMERA_WIDTH, face_landmarks.landmark[291].y * CAMERA_HEIGHT],
        [face_landmarks.landmark[61].x * CAMERA_WIDTH, face_landmarks.landmark[61].y * CAMERA_HEIGHT]
    ], dtype=np.float64)

    # центр лица
    left_eye = np.array([face_landmarks.landmark[263].x * CAMERA_WIDTH, face_landmarks.landmark[263].y * CAMERA_HEIGHT])
    right_eye = np.array([face_landmarks.landmark[33].x * CAMERA_WIDTH, face_landmarks.landmark[33].y * CAMERA_HEIGHT])
    nose_tip = np.array([face_landmarks.landmark[1].x * CAMERA_WIDTH, face_landmarks.landmark[1].y * CAMERA_HEIGHT])
    face_center = (left_eye + right_eye + nose_tip) / 3

    # смещение лица относительно центра кадра
    offset = face_center[0] - CAMERA_WIDTH / 2

    # нормализация по межглазному расстоянию
    inter_eye_dist = np.linalg.norm(right_eye - left_eye)
    image_points_centered = (image_points - face_center) / inter_eye_dist

    # solvePnP
    camera_matrix = np.array([[1,0,0],[0,1,0],[0,0,1]], dtype=np.float64)
    dist_coeffs = np.zeros((4,1))
    success, rotation_vector, _ = cv2.solvePnP(model_points, image_points_centered, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE)
    rmat, _ = cv2.Rodrigues(rotation_vector)

    # углы
    sy = math.sqrt(rmat[0,0]**2 + rmat[1,0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = math.degrees(math.atan2(rmat[2,1], rmat[2,2]))
        yaw = math.degrees(math.atan2(-rmat[2,0], sy))
        roll = math.degrees(math.atan2(rmat[1,0], rmat[0,0]))
    else:
        pitch = math.degrees(math.atan2(-rmat[1,2], rmat[1,1]))
        yaw = math.degrees(math.atan2(-rmat[2,0], sy))
        roll = 0

    # корректировка yaw с учётом положения лица в кадре
    yaw += offset / (CAMERA_WIDTH / 2) * 20  # ±20° за край кадра

    angles = np.array([pitch, yaw, roll])
    if prev_angles is not None:
        angles = alpha * angles + (1 - alpha) * prev_angles

    return angles, rmat


In [5]:
# -----------------------------
# Функция вывода значений координат основных точек и углов поверх потокового изображения
# -----------------------------

def draw_debug_text(
    canvas,
    left_iris_xy,
    right_iris_xy,
    left_eye_corners,
    right_eye_corners,
    face_bbox_xyxy,
    face_center_xy,
    angles,
    origin=(10, 30),
    dy=22
):
    """
    angles = (yaw, pitch, roll) в градусах
    """

    x0, y0 = origin
    yaw, pitch, roll = angles

    cv2.putText(canvas, f"Left iris: {left_iris_xy}",
                (x0, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.putText(canvas, f"Left eye corners: {left_eye_corners[0]}, {left_eye_corners[1]}",
                (x0, y0 + dy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.putText(canvas, f"Right iris: {right_iris_xy}",
                (x0, y0 + 2*dy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

    cv2.putText(canvas, f"Right eye corners: {right_eye_corners[0]}, {right_eye_corners[1]}",
                (x0, y0 + 3*dy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

    cv2.putText(canvas, f"Face bbox: {face_bbox_xyxy}",
                (x0, y0 + 4*dy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.putText(canvas, f"Face center: {face_center_xy}",
                (x0, y0 + 5*dy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.putText(canvas, f"Head angles (deg): yaw={yaw:.1f}, pitch={pitch:.1f}, roll={roll:.1f}",
                (x0, y0 + 6*dy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)


In [6]:
# -----------------------------
# Функции рисования сетки поверх потового изображения
# -----------------------------

def draw_grid_with_points_all_red(
    image,
    cols=5,
    rows=5,
    margin=50,
    point_radius=20,
    line_color=(200, 200, 200),
    point_color=(0, 0, 255),
    line_thickness=1
):
    """
    Рисует сетку и красные точки на пересечениях.

    image         : np.ndarray (BGR)
    cols          : количество столбцов
    rows          : количество строк
    margin        : отступ от краёв (в пикселях)
    point_radius  : радиус круга (5 => диаметр 10)
    """

    h, w = image.shape[:2]

    # Рабочая область
    x_start = margin
    x_end = w - margin
    y_start = margin
    y_end = h - margin

    if cols < 2 or rows < 2:
        raise ValueError("cols и rows должны быть >= 2")

    dx = (x_end - x_start) / (cols - 1)
    dy = (y_end - y_start) / (rows - 1)

    # --- Вертикальные линии ---
    for c in range(cols):
        x = int(x_start + c * dx)
        cv2.line(image, (x, y_start), (x, y_end), line_color, line_thickness)

    # --- Горизонтальные линии ---
    for r in range(rows):
        y = int(y_start + r * dy)
        cv2.line(image, (x_start, y), (x_end, y), line_color, line_thickness)

    # --- Красные точки на пересечениях ---
    for r in range(rows):
        for c in range(cols):
            x = int(x_start + c * dx)
            y = int(y_start + r * dy)
            cv2.circle(image, (x, y), point_radius, point_color, -1)

    return image


def draw_grid_with_points(
    image,
    cols,
    rows,
    highlight_row=None,
    highlight_col=None,
    highlight_color=(0, 255, 0),
    margin=50,
    point_radius=20,
    line_color=(200, 200, 200),
    point_color=(0, 0, 255),
    line_thickness=1
):
    """
    Рисует сетку и точки на пересечениях.
    По умолчанию все точки красные (point_color).
    Если задано highlight_row/highlight_col — точка в этом пересечении рисуется highlight_color.

    Параметры:
      - cols, rows: количество столбцов и строк (>=2)
      - highlight_row, highlight_col: индекс строки/столбца (0-based). Можно None.
      - highlight_color: цвет выделенной точки (BGR)
      - margin: отступ крайних точек от края (пиксели)
      - point_radius: радиус точки (5 => диаметр 10)
    """
    h, w = image.shape[:2]

    if cols < 2 or rows < 2:
        raise ValueError("cols и rows должны быть >= 2")

    # Рабочая область
    x_start = margin
    x_end = w - margin
    y_start = margin
    y_end = h - margin

    dx = (x_end - x_start) / (cols - 1)
    dy = (y_end - y_start) / (rows - 1)

    # Линии сетки
    for c in range(cols):
        x = int(x_start + c * dx)
        cv2.line(image, (x, y_start), (x, y_end), line_color, line_thickness)

    for r in range(rows):
        y = int(y_start + r * dy)
        cv2.line(image, (x_start, y), (x_end, y), line_color, line_thickness)

    # Точки на пересечениях
    for r in range(rows):
        for c in range(cols):
            x = int(x_start + c * dx)
            y = int(y_start + r * dy)

            color = point_color
            if (highlight_row is not None) and (highlight_col is not None):
                if (r == highlight_row) and (c == highlight_col):
                    color = highlight_color

            cv2.circle(image, (x, y), point_radius, color, -1)

    return image

In [7]:
# -----------------------------
# Основная функция запуска: обработка потового изображения, вывод значений на экран
# -----------------------------

def run_real_time_face_tracking_3D(alpha=0.3, duration_sec=10):

    mp_face_mesh = mp.solutions.face_mesh
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("ERROR: cannot open camera")
        return

    prev_angles = None
    start_time = time.time()  # ⏱ старт таймера

    with mp_face_mesh.FaceMesh(
                                static_image_mode=False,
                                max_num_faces=1,
                                refine_landmarks=True,
                                min_detection_confidence=0.5,
                                min_tracking_confidence=0.5
                              ) as face_mesh:

        while True:

            # ⏱ проверка времени
            if time.time() - start_time >= duration_sec:
                break

            ret, frame = cap.read()
            
            if not ret:
                continue

            frame = cv2.flip(frame, 1)
            h, w, _ = frame.shape

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_frame)

            canvas = frame.copy()

            #draw_grid_with_points_all_red(canvas, cols=6, rows=4)
            draw_grid_with_points(canvas, cols=6, rows=4, highlight_row=2, highlight_col=4, highlight_color=(0,255,0))
            cv2.imshow("Grid", canvas)

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                # --- ключевые точки лица ---
                kp = get_face_keypoints(face_landmarks, w, h)

                # --- зрачки ---
                left_pupil = np.array(get_pupil_center(face_landmarks, LEFT_EYE, w, h))
                right_pupil = np.array(get_pupil_center(face_landmarks, RIGHT_EYE, w, h))

                left_iris_xy = (int(left_pupil[0]), int(left_pupil[1]))
                right_iris_xy = (int(right_pupil[0]), int(right_pupil[1]))

                # --- углы глаз ---
                left_eye_corners = (
                                        (int(face_landmarks.landmark[33].x * w), int(face_landmarks.landmark[33].y * h)),
                                        (int(face_landmarks.landmark[133].x * w), int(face_landmarks.landmark[133].y * h))
                                    )
                right_eye_corners = (
                                        (int(face_landmarks.landmark[362].x * w), int(face_landmarks.landmark[362].y * h)),
                                        (int(face_landmarks.landmark[263].x * w), int(face_landmarks.landmark[263].y * h))
                                    )

                # --- bbox лица ---
                xs = [int(lm.x * w) for lm in face_landmarks.landmark]
                ys = [int(lm.y * h) for lm in face_landmarks.landmark]

                min_x, max_x = min(xs), max(xs)
                min_y, max_y = min(ys), max(ys)

                face_bbox_xyxy = (min_x, min_y, max_x, max_y)
                face_center_xy = ((min_x + max_x) // 2, (min_y + max_y) // 2)

                # --- углы головы ---
                angles, rmat = estimate_head_pose_3d (face_landmarks, prev_angles, alpha=alpha)
                prev_angles = angles

                # --- отрисовка твоих элементов ---
                try:
                    draw_face_keypoints(canvas, kp)
                    draw_eye_iris(canvas, face_landmarks, w, h)
                except Exception:
                    pass

                # --- DEBUG overlay ---
                draw_debug_text(
                    canvas=canvas,
                    left_iris_xy=left_iris_xy,
                    right_iris_xy=right_iris_xy,
                    left_eye_corners=left_eye_corners,
                    right_eye_corners=right_eye_corners,
                    face_bbox_xyxy=face_bbox_xyxy,
                    face_center_xy=face_center_xy,
                    angles=angles
                )

            cv2.imshow("Face tracking 3D", canvas)

            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()


In [8]:
run_real_time_face_tracking_3D(0.8, 10)

I0000 00:00:1768465006.674215       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro


In [9]:
# -----------------------------
# Функция для калибровки отслеживания изображения
# -----------------------------
class CalibrationExcelWriter:
    def __init__(self):
        self.wb = Workbook()
        self.ws = self.wb.active
        self.ws.title = "calibration"

        # Заголовки
        self.ws.append([
            "grid_x", "grid_y",
            "left_pupil_x", "left_pupil_y",
            "left_corner_left_x", "left_corner_left_y",
            "left_corner_right_x", "left_corner_right_y",
            "right_pupil_x", "right_pupil_y",
            "right_corner_left_x", "right_corner_left_y",
            "right_corner_right_x", "right_corner_right_y",

            # губы (в v2 у тебя они уже есть в kp)
            "mouth_left_x", "mouth_left_y",
            "mouth_right_x", "mouth_right_y",
            "upper_lip_x", "upper_lip_y",
            "lower_lip_x", "lower_lip_y",

            "face_center_x", "face_center_y",
            "yaw", "pitch", "roll",
            "timestamp"
        ])

    def append_row(self, values):
        self.ws.append(values)

    def save_to_desktop(self, filename="calibration.xlsx"):
        desktop = os.path.join(os.path.expanduser("~"), "Desktop")
        os.makedirs(desktop, exist_ok=True)
        path = os.path.join(desktop, filename)
        self.wb.save(path)
        return path


In [10]:
# -----------------------------
# Функция для записи координат основных точек и углов в отдельный файл
# -----------------------------

def record_calibration_frame_if_recording(
                                            writer: CalibrationExcelWriter,
                                            is_recording: bool,        # True только когда точка зелёная (phase == "record")
                                            grid_xy,                   # (grid_x, grid_y) — координаты зелёной точки сетки
                                            kp,                        # dict из get_face_keypoints(...) (как в Untitled v2)
                                            face_center_xy,            # (x, y)
                                            angles,                    # (yaw, pitch, roll)
                                            timestamp: float
                                         ):
    if not is_recording:
        return

 
    
    (gx, gy) = grid_xy

    # Левый глаз
    (lp_x, lp_y) = kp["left_eye"]["pupil"]
    (lcL_x, lcL_y) = kp["left_eye"]["left"]
    (lcR_x, lcR_y) = kp["left_eye"]["right"]

    # Правый глаз
    (rp_x, rp_y) = kp["right_eye"]["pupil"]
    (rcL_x, rcL_y) = kp["right_eye"]["left"]
    (rcR_x, rcR_y) = kp["right_eye"]["right"]

    # Губы и центр лица
    mouth_left = kp["mouth"][0]
    mouth_right = kp["mouth"][1]
    upper_lip = kp["upper_lip"]
    lower_lip = kp["lower_lip"]
    (fc_x, fc_y) = face_center_xy

    # Углы поворота
    (yaw, pitch, roll) = angles

    writer.append_row([
        gx, gy,
        lp_x, lp_y,
        lcL_x, lcL_y,
        lcR_x, lcR_y,
        rp_x, rp_y,
        rcL_x, rcL_y,
        rcR_x, rcR_y,
        mouth_left[0], mouth_left[1],
        mouth_right[0], mouth_right[1],
        upper_lip[0], upper_lip[1],
        lower_lip[0], lower_lip[1],
        fc_x, fc_y,
        float(yaw), float(pitch), float(roll),
        float(timestamp)
    ])


In [11]:
# -----------------------------
# Функция запуска калибровки: обработка потового изображения, определение основых точек и углов, запись в файл
# -----------------------------
def run_calibration_face_tracking_3D(
    alpha=0.3,
    cols=6,
    rows=4,
    initial_delay_sec=2.0,   # все точки красные
    ready_time_sec=1.0,      # одна точка жёлтая
    record_time_sec=1.5,     # та же точка зелёная + пишем в файл КАЖДЫЙ кадр
    camera_index=0,
    margin=50
):
    """
    Калибровка по сетке (row-major: по строке слева направо, затем следующая строка).
    Фазы:
      initial_delay_sec: все точки красные
      ready_time_sec: текущая точка жёлтая
      record_time_sec: текущая точка зелёная, запись данных КАЖДЫЙ кадр

    По завершении всех точек сохраняет Excel:
      ~/Desktop/calibration.xlsx

    Требуются уже существующие в Untitled v2.ipynb:
      - draw_grid_with_points(...)
      - get_face_keypoints(face_landmarks, w, h)  -> kp (включая eyes + mouth + lips)
      - estimate_head_pose_3d(face_landmarks, prev_angles, alpha=alpha) -> angles
      - CalibrationExcelWriter
      - record_calibration_frame_if_recording
    """

    def grid_point_xy(frame_w, frame_h, cols_, rows_, row_, col_, margin_=50):
        # координаты пересечения сетки в пикселях (row/col — 0-based)
        x_start, x_end = margin_, frame_w - margin_
        y_start, y_end = margin_, frame_h - margin_
        dx = (x_end - x_start) / (cols_ - 1)
        dy = (y_end - y_start) / (rows_ - 1)
        x = int(x_start + col_ * dx)
        y = int(y_start + row_ * dy)
        return (x, y)

    # --- Excel writer ---
    writer = CalibrationExcelWriter()

    mp_face_mesh = mp.solutions.face_mesh
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print("ERROR: cannot open camera")
        return None

    prev_angles = None

    # --- калибровочный автомат ---
    phase = "initial"          # "initial" -> "ready" -> "record" -> "ready" -> ...
    phase_start = time.time()
    current_r, current_c = 0, 0

    def advance_point():
        nonlocal current_r, current_c
        current_c += 1
        if current_c >= cols:
            current_c = 0
            current_r += 1

    with mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as face_mesh:

        while True:
            ret, frame = cap.read()
            if not ret:
                continue

            frame = cv2.flip(frame, 1)
            h, w = frame.shape[:2]

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_frame)

            canvas = frame.copy()

            now = time.time()
            elapsed = now - phase_start

            # --- определяем highlight для сетки ---
            highlight_row = None
            highlight_col = None
            highlight_color = None

            if phase == "initial":
                # всё красное
                if elapsed >= initial_delay_sec:
                    phase = "ready"
                    phase_start = now
            else:
                # "ready" или "record"
                highlight_row, highlight_col = current_r, current_c
                if phase == "ready":
                    highlight_color = (0, 255, 255)  # жёлтый (BGR)
                    if elapsed >= ready_time_sec:
                        phase = "record"
                        phase_start = now
                elif phase == "record":
                    highlight_color = (0, 255, 0)    # зелёный (BGR)
                    if elapsed >= record_time_sec:
                        # закончили запись этой точки -> следующая
                        advance_point()
                        if current_r >= rows:
                            # все точки пройдены
                            break
                        phase = "ready"
                        phase_start = now

            # --- рисуем сетку ---
            if highlight_row is None:
                draw_grid_with_points(canvas, cols=cols, rows=rows, margin=margin)
            else:
                draw_grid_with_points(
                    canvas,
                    cols=cols,
                    rows=rows,
                    highlight_row=highlight_row,
                    highlight_col=highlight_col,
                    highlight_color=highlight_color,
                    margin=margin
                )

            # --- координаты текущей точки сетки (для записи/оверлея) ---
            grid_xy = grid_point_xy(w, h, cols, rows, current_r, current_c, margin_=margin)

            # --- если нашли лицо: считаем kp/angles/face_center и пишем при record ---
            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                # kp уже содержит все нужные координаты (как ты сказал — в v2 это уже есть)
                kp = get_face_keypoints(face_landmarks, w, h)

                # углы головы
                angles, _rmat = estimate_head_pose_3d(face_landmarks, prev_angles, alpha=alpha)
                prev_angles = angles

                # центр лица (в v2 он у тебя уже есть; но если нет — считаем безопасно)
                # стараемся взять готовое имя из kp, иначе fallback:
                face_center_xy = None
                if isinstance(kp, dict) and ("face_center" in kp):
                    face_center_xy = kp["face_center"]
                if face_center_xy is None:
                    xs = [int(lm.x * w) for lm in face_landmarks.landmark]
                    ys = [int(lm.y * h) for lm in face_landmarks.landmark]
                    face_center_xy = ((min(xs) + max(xs)) // 2, (min(ys) + max(ys)) // 2)

                # запись КАЖДЫЙ кадр во время зелёной фазы
                record_calibration_frame_if_recording(
                    writer=writer,
                    is_recording=(phase == "record"),
                    grid_xy=grid_xy,
                    kp=kp,
                    face_center_xy=face_center_xy,
                    angles=angles,
                    timestamp=now
                )

                # небольшая подсказка на экране
                cv2.putText(
                    canvas,
                    f"Phase: {phase} | point r={current_r} c={current_c} | grid={grid_xy}",
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 255),
                    2
                )
            else:
                cv2.putText(
                    canvas,
                    f"No face | Phase: {phase} | point r={current_r} c={current_c} | grid={grid_xy}",
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 255),
                    2
                )

            cv2.imshow("Calibration", canvas)

            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

    # --- сохраняем Excel на Desktop после прохода всех точек ---
    path = writer.save_to_desktop("calibration.xlsx")
    print("Saved:", path)
    return path


In [13]:
# -----------------------------
# Функция загрузки даных полученных в результате калибровки
# Функции обучения различных моделей
# -----------------------------

def _load_calibration_xy_features(filename="calibration.xlsx"):
    """
    Загружает calibration.xlsx с Desktop.
    Возвращает: X (DataFrame), yx (Series), yy (Series), feature_cols (list)
    """

    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    path = os.path.join(desktop, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")

    df = pd.read_excel(path)

    yx = df["grid_x"].astype(float)
    yy = df["grid_y"].astype(float)

    X = df.drop(columns=["grid_x", "grid_y", "timestamp"], errors="ignore")
    # На всякий случай: уберём нечисловые
    X = X.select_dtypes(include=["number"]).copy()

    return X, yx, yy, list(X.columns)


def _fit_two_models(model_factory, X, yx, yy):
    """
    Обучает две модели: для grid_x и grid_y.
    """
    mx = model_factory()
    my = model_factory()
    mx.fit(X, yx)
    my.fit(X, yy)
    return mx, my


# ------------------------------------------------------------
# 0) Linear Regression (baseline)
# ------------------------------------------------------------
def train_linear_regression_from_calibration(filename="calibration.xlsx"):
    """
    Базовая линейная регрессия (без регуляризации).
    Возвращает: model_x, model_y, feature_cols
    """

    X, yx, yy, feature_cols = _load_calibration_xy_features(filename)

    def factory():
        return LinearRegression()

    mx, my = _fit_two_models(factory, X, yx, yy)
    return mx, my, feature_cols


# ------------------------------------------------------------
# 1) Ridge
# ------------------------------------------------------------
def train_ridge_from_calibration(filename="calibration.xlsx", alpha=1.0):
    """
    Возвращает: model_x, model_y, feature_cols
    """

    X, yx, yy, feature_cols = _load_calibration_xy_features(filename)

    def factory():
        return Ridge(alpha=alpha, random_state=42)

    mx, my = _fit_two_models(factory, X, yx, yy)
    return mx, my, feature_cols


# ------------------------------------------------------------
# 2) ElasticNet
# ------------------------------------------------------------
def train_elasticnet_from_calibration(
    filename="calibration.xlsx",
    alpha=0.1,
    l1_ratio=0.5,
    max_iter=10000
):
    """
    Возвращает: model_x, model_y, feature_cols
    """

    X, yx, yy, feature_cols = _load_calibration_xy_features(filename)

    def factory():
        return ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=max_iter,
            random_state=42
        )

    mx, my = _fit_two_models(factory, X, yx, yy)
    return mx, my, feature_cols


# ------------------------------------------------------------
# 3) RandomForestRegressor
# ------------------------------------------------------------
def train_random_forest_from_calibration(
    filename="calibration.xlsx",
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2
):
    """
    Возвращает: model_x, model_y, feature_cols
    """

    X, yx, yy, feature_cols = _load_calibration_xy_features(filename)

    def factory():
        return RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
            n_jobs=-1
        )

    mx, my = _fit_two_models(factory, X, yx, yy)
    return mx, my, feature_cols


# ------------------------------------------------------------
# 4) Polynomial Regression (degree=2)
# ------------------------------------------------------------
def train_poly2_from_calibration(filename="calibration.xlsx", include_bias=False):
    """
    Полиномиальная регрессия 2 степени.
    Возвращает: pipeline_x, pipeline_y, feature_cols

    Pipeline = PolynomialFeatures(degree=2) + LinearRegression
    """

    X, yx, yy, feature_cols = _load_calibration_xy_features(filename)

    def factory():
        return Pipeline([
            ("poly", PolynomialFeatures(degree=2, include_bias=include_bias)),
            ("lin", LinearRegression())
        ])

    mx, my = _fit_two_models(factory, X, yx, yy)
    return mx, my, feature_cols


# ------------------------------------------------------------
# 5) Polynomial Regression (degree=3)
# ------------------------------------------------------------
def train_poly3_from_calibration(filename="calibration.xlsx", include_bias=False):
    """
    Полиномиальная регрессия 3 степени.
    Возвращает: pipeline_x, pipeline_y, feature_cols
    """

    X, yx, yy, feature_cols = _load_calibration_xy_features(filename)

    def factory():
        return Pipeline([
            ("poly", PolynomialFeatures(degree=3, include_bias=include_bias)),
            ("lin", LinearRegression())
        ])

    mx, my = _fit_two_models(factory, X, yx, yy)
    return mx, my, feature_cols


# ------------------------------------------------------------
# 6) LightGBM
# ------------------------------------------------------------
def train_lightgbm_from_calibration(
    filename="calibration.xlsx",
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1
):
    """
    Требует установленный lightgbm:
      pip install lightgbm

    Возвращает: model_x, model_y, feature_cols
    """
    X, yx, yy, feature_cols = _load_calibration_xy_features(filename)

    def factory():
        return LGBMRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            num_leaves=num_leaves,
            max_depth=max_depth,
            random_state=42
        )

    mx, my = _fit_two_models(factory, X, yx, yy)
    return mx, my, feature_cols


In [14]:
# -----------------------------
# Основная фукнция обучения моделей 
# -----------------------------

def compare_models_on_calibration(
    filename="calibration.xlsx",
    test_size=0.2,
    random_state=42,
    n_speed_samples=2000
):

    # --- загрузка данных ---
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    path = os.path.join(desktop, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")

    df = pd.read_excel(path)

    y = df[["grid_x", "grid_y"]].astype(float)
    X = df.drop(columns=["grid_x", "grid_y", "timestamp"], errors="ignore")
    X = X.select_dtypes(include=["number"]).copy()

    feature_cols = list(X.columns)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # --- фабрики моделей (две модели: для X и Y) ---

    def fit_two(factory):
        mx = factory()
        my = factory()
        mx.fit(X_train, y_train["grid_x"])
        my.fit(X_train, y_train["grid_y"])
        return mx, my

    models = []

    # Linear Regression (baseline)
    models.append(("LinearRegression", lambda: LinearRegression()))

    # Ridge
    models.append(("Ridge(alpha=1.0)", lambda: Ridge(alpha=1.0, random_state=random_state)))

    # ElasticNet
    models.append(("ElasticNet(alpha=0.1,l1=0.5)", lambda: ElasticNet(
        alpha=0.1, l1_ratio=0.5, max_iter=10000, random_state=random_state
    )))

    # RandomForest
    models.append(("RandomForest(n=400)", lambda: RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, random_state=random_state, n_jobs=-1
    )))

    # Poly2
    models.append(("Poly2(Linear)", lambda: Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("lin", LinearRegression())
    ])))

    # Poly3
    models.append(("Poly3(Linear)", lambda: Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("lin", LinearRegression())
    ])))

    # LightGBM
    models.append(("LightGBM", lambda: LGBMRegressor(
            n_estimators=600, learning_rate=0.05, num_leaves=31, max_depth=-1, random_state=random_state
        )))

    # --- оценка ---
    results = []

    # Для скорости: возьмём небольшой батч из X_test
    X_speed = X_test.copy()
    if len(X_speed) == 0:
        raise ValueError("Пустая test-выборка. Увеличь данные или измени test_size.")
    if len(X_speed) > n_speed_samples:
        X_speed = X_speed.iloc[:n_speed_samples]

    for name, factory in models:
        mx, my = fit_two(factory)

        pred_x = mx.predict(X_test)
        pred_y = my.predict(X_test)

        true_x = y_test["grid_x"].to_numpy()
        true_y = y_test["grid_y"].to_numpy()

        # ошибки
        mae_x = mean_absolute_error(true_x, pred_x)
        mae_y = mean_absolute_error(true_y, pred_y)

        rmse_x = np.sqrt(mean_squared_error(true_x, pred_x))
        rmse_y = np.sqrt(mean_squared_error(true_y, pred_y))

        # 2D ошибка в пикселях
        err_2d = np.sqrt((pred_x - true_x) ** 2 + (pred_y - true_y) ** 2)
        mae_2d = float(np.mean(err_2d))
        rmse_2d = float(np.sqrt(np.mean(err_2d ** 2)))
        p95_2d = float(np.percentile(err_2d, 95))

        r2_x = r2_score(true_x, pred_x)
        r2_y = r2_score(true_y, pred_y)

        # скорость инференса (мс на кадр: предсказать x и y)
        t0 = time.perf_counter()
        _ = mx.predict(X_speed)
        _ = my.predict(X_speed)
        t1 = time.perf_counter()
        ms_per_frame = (t1 - t0) * 1000.0 / len(X_speed)

        results.append({
            "model": name,
            "mae_x": mae_x,
            "mae_y": mae_y,
            "rmse_x": rmse_x,
            "rmse_y": rmse_y,
            "mae_2d_px": mae_2d,
            "rmse_2d_px": rmse_2d,
            "p95_2d_px": p95_2d,
            "r2_x": r2_x,
            "r2_y": r2_y,
            "ms_per_frame": ms_per_frame,
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": len(feature_cols),
        })

    res_df = pd.DataFrame(results)

    # сортируем: сначала по средней 2D ошибке, потом по скорости
    res_df = res_df.sort_values(["mae_2d_px", "ms_per_frame"], ascending=[True, True]).reset_index(drop=True)

    return res_df


In [20]:
df_scores = compare_models_on_calibration()
#df_scores
df_scores[['model','rmse_x','rmse_y']]

/Users/work/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.122e+07, tolerance: 5.409e+04
  model = cd_fast.enet_coordinate_descent(
/Users/work/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.618e+06, tolerance: 1.679e+04
  model = cd_fast.enet_coordinate_descent(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000644 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5220
[LightGBM] [Info] Number of data points in the train set: 1395, number of used features: 25
[LightGBM] [Info] Start training from score 962.217921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000279 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5220
[LightGBM] [Info] Number of data points in the train set: 1395, number of used features: 25
[LightGBM] [Info] Start training from score 546.322581


,model,rmse_x,rmse_y
0,Poly2(Linear),91.505097,63.785543
1,LightGBM,162.718544,58.884171
2,"ElasticNet(alpha=0.1,l1=0.5)",124.723513,110.648618
3,Ridge(alpha=1.0),124.555242,110.157764
4,LinearRegression,124.574788,110.156958
5,RandomForest(n=400),216.464991,71.034076
6,Poly3(Linear),250.121388,140.484191


In [21]:
def run_real_time_face_tracking_3D(
    alpha=0.3,
    duration_sec=60,
    use_online_gaze=True,
    gaze_model="linear",          # "linear" | "ridge" | "elasticnet" | "rf" | "poly2" | "poly3" | "lightgbm"
    model_params=None,            # dict с параметрами выбранной модели (опционально)
    filename="calibration.xlsx"   # файл на Desktop
):
    """
    Реальный трекинг + онлайн определение точки взгляда (по выбранной модели).

    gaze_model варианты:
      - "linear"    : LinearRegression
      - "ridge"     : Ridge
      - "elasticnet": ElasticNet
      - "rf"        : RandomForestRegressor
      - "poly2"     : PolynomialFeatures(deg=2) + LinearRegression
      - "poly3"     : PolynomialFeatures(deg=3) + LinearRegression
      - "lightgbm"  : LGBMRegressor (если установлен lightgbm)

    model_params — словарь параметров конкретной модели (если None, используются дефолты).
    Требуются функции из твоего v2:
      - get_face_keypoints(face_landmarks, w, h) -> kp
      - estimate_head_pose_3d(face_landmarks, prev_angles, alpha=alpha) -> (angles, rmat)
      - draw_grid_with_points(...) (если рисуешь сетку)
    """

    model_params = model_params or {}

    # ---------- 1) обучение выбранной модели из calibration.xlsx (1 раз) ----------
    model_x = None
    model_y = None
    feature_cols = None

    def make_factory(kind, params):
        kind = kind.lower().strip()

        if kind == "linear":
            return lambda: LinearRegression(**params)

        if kind == "ridge":
            # дефолт
            p = {"alpha": 1.0, "random_state": 42}
            p.update(params)
            return lambda: Ridge(**p)

        if kind == "elasticnet":
            p = {"alpha": 0.1, "l1_ratio": 0.5, "max_iter": 10000, "random_state": 42}
            p.update(params)
            return lambda: ElasticNet(**p)

        if kind == "rf":
            p = {"n_estimators": 400, "min_samples_leaf": 2, "random_state": 42, "n_jobs": -1}
            p.update(params)
            return lambda: RandomForestRegressor(**p)

        if kind == "poly2":
            # params: include_bias (bool) можно передать, остальные игнорим
            include_bias = bool(params.get("include_bias", False))
            return lambda: Pipeline([
                                        ("poly", PolynomialFeatures(degree=2, include_bias=include_bias)),
                                        ("lin", LinearRegression())
                                    ])

        if kind == "poly3":
            include_bias = bool(params.get("include_bias", False))
            return lambda: Pipeline([
                                        ("poly", PolynomialFeatures(degree=3, include_bias=include_bias)),
                                        ("lin", LinearRegression())
                                    ])

        if kind == "lightgbm":
            p = {"n_estimators": 600, "learning_rate": 0.05, "num_leaves": 31, "max_depth": -1, "random_state": 42}
            p.update(params)
            return lambda: LGBMRegressor(**p)


        raise ValueError(f"Неизвестная модель gaze_model='{kind}'")

    def train_models_from_desktop_excel():
        desktop = os.path.join(os.path.expanduser("~"), "Desktop")
        path = os.path.join(desktop, filename)
        if not os.path.exists(path):
            return None, None, None

        df = pd.read_excel(path)

        yx = df["grid_x"].astype(float)
        yy = df["grid_y"].astype(float)

        X = df.drop(columns=["grid_x", "grid_y", "timestamp"], errors="ignore")
        X = X.select_dtypes(include=["number"]).copy()  # только числа

        factory = make_factory(gaze_model, model_params)

        mx = factory()
        my = factory()
        mx.fit(X, yx)
        my.fit(X, yy)

        return mx, my, list(X.columns)

    def build_feature_row(kp, face_center_xy, angles, cols_order):
        """
        Собирает признаки в DataFrame с колонками cols_order (чтобы совпадало с обучением).
        """
        yaw, pitch, roll = angles

        lp = kp["left_eye"]["pupil"]
        ll = kp["left_eye"]["left"]
        lr = kp["left_eye"]["right"]

        rp = kp["right_eye"]["pupil"]
        rl = kp["right_eye"]["left"]
        rr = kp["right_eye"]["right"]

        mouth_left = kp["mouth"][0]
        mouth_right = kp["mouth"][1]
        upper_lip = kp["upper_lip"]
        lower_lip = kp["lower_lip"]

        fc_x, fc_y = face_center_xy

        feat_map = {
            "left_pupil_x": lp[0], "left_pupil_y": lp[1],
            "left_corner_left_x": ll[0], "left_corner_left_y": ll[1],
            "left_corner_right_x": lr[0], "left_corner_right_y": lr[1],

            "right_pupil_x": rp[0], "right_pupil_y": rp[1],
            "right_corner_left_x": rl[0], "right_corner_left_y": rl[1],
            "right_corner_right_x": rr[0], "right_corner_right_y": rr[1],

            "mouth_left_x": mouth_left[0], "mouth_left_y": mouth_left[1],
            "mouth_right_x": mouth_right[0], "mouth_right_y": mouth_right[1],
            "upper_lip_x": upper_lip[0], "upper_lip_y": upper_lip[1],
            "lower_lip_x": lower_lip[0], "lower_lip_y": lower_lip[1],

            "face_center_x": fc_x, "face_center_y": fc_y,

            "yaw": float(yaw), "pitch": float(pitch), "roll": float(roll),
        }

        row = {c: float(feat_map.get(c, 0.0)) for c in cols_order}
        return pd.DataFrame([row], columns=cols_order)

    if use_online_gaze:
        model_x, model_y, feature_cols = train_models_from_desktop_excel()
        if model_x is None:
            print(f"WARN: ~/Desktop/{filename} не найден — онлайн-инференс отключён.")
            use_online_gaze = False
        else:
            print(f"OK: Онлайн-инференс включён. Модель: {gaze_model}")

    # ---------- 2) камера + facemesh ----------
    mp_face_mesh = mp.solutions.face_mesh
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("ERROR: cannot open camera")
        return

    prev_angles = None
    start_time = time.time()

    # сглаживание предсказанной точки (EMA)
    pred_smoothed = None
    pred_alpha = 0.35

    with mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as face_mesh:

        while True:
            if duration_sec is not None and (time.time() - start_time >= duration_sec):
                break

            ret, frame = cap.read()
            if not ret:
                continue

            frame = cv2.flip(frame, 1)
            h, w = frame.shape[:2]

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_frame)

            canvas = frame.copy()

            try:
                draw_grid_with_points(canvas, cols=6, rows=4)
            except Exception:
                pass

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                kp = get_face_keypoints(face_landmarks, w, h)

                angles, _rmat = estimate_head_pose_3d(face_landmarks, prev_angles, alpha=alpha)
                prev_angles = angles

                if isinstance(kp, dict) and ("face_center" in kp):
                    face_center_xy = kp["face_center"]
                else:
                    xs = [int(lm.x * w) for lm in face_landmarks.landmark]
                    ys = [int(lm.y * h) for lm in face_landmarks.landmark]
                    face_center_xy = ((min(xs) + max(xs)) // 2, (min(ys) + max(ys)) // 2)

                # ---------- 3) онлайн-инференс точки взгляда ----------
                if use_online_gaze:
                    Xrow = build_feature_row(kp, face_center_xy, angles, feature_cols)
                    pred_x = float(model_x.predict(Xrow)[0])
                    pred_y = float(model_y.predict(Xrow)[0])

                    if pred_smoothed is None:
                        pred_smoothed = (pred_x, pred_y)
                    else:
                        px, py = pred_smoothed
                        px = px * (1 - pred_alpha) + pred_x * pred_alpha
                        py = py * (1 - pred_alpha) + pred_y * pred_alpha
                        pred_smoothed = (px, py)

                    gx, gy = int(pred_smoothed[0]), int(pred_smoothed[1])
                    cv2.circle(canvas, (gx, gy), 8, (255, 0, 255), -1)
                    cv2.putText(
                        canvas,
                        f"Gaze[{gaze_model}]: ({gx},{gy})",
                        (10, 55),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (255, 0, 255),
                        2
                    )

                yaw, pitch, roll = angles
                cv2.putText(
                    canvas,
                    f"yaw={yaw:.1f} pitch={pitch:.1f} roll={roll:.1f}",
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 255),
                    2
                )

            cv2.imshow("Face tracking 3D", canvas)

            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
